# Preparación de Silver: lectura ejecutiva, calidad estructural y base para Gold

Este notebook convierte los Parquet de `datalake_silver` en una lectura guiada. Primero inventaria los archivos, luego revisa el grano real de los registros, valida si los nulos son estructurales o reales y termina con un CSV listo para EDA y diseño Gold.

La intención ejecutiva es responder dos preguntas: qué parte del contenido ya es estable para modelado y qué parte debe quedarse como trazabilidad o separarse por entidad.

In [1]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

def find_project_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'datalake_silver').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()
SILVER_DIR = PROJECT_ROOT / 'datalake_silver'
OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SILVER_EDA_CSV = OUTPUT_DIR / 'silver_eda.csv'

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'SILVER_DIR = {SILVER_DIR}')
print(f'SILVER_EDA_CSV = {SILVER_EDA_CSV}')

PROJECT_ROOT = /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming
SILVER_DIR = /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming/datalake_silver
SILVER_EDA_CSV = /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming/notebooks/output/silver_eda.csv


In [2]:
silver_files = sorted(SILVER_DIR.glob('*.parquet'))
print(f'Parquet files found: {len(silver_files)}')
display(pd.DataFrame({'silver_parquet': [f.name for f in silver_files]}))

Parquet files found: 13


,silver_parquet
0,noticias_processed_20260530_012014.parquet
1,noticias_processed_20260601_204305.parquet
2,noticias_processed_20260601_204418.parquet
3,noticias_processed_20260601_204419.parquet
4,noticias_processed_20260602_014739.parquet
5,noticias_processed_20260602_223238.parquet
6,noticias_processed_20260602_223251.parquet
7,tweets_processed_20260530_012013.parquet
8,tweets_processed_20260601_204305.parquet
9,tweets_processed_20260601_204414.parquet


## 1. Inventario de archivos y volumen

Se confirma cuántos Parquet existen, qué familias representan y desde dónde se va a consolidar la tabla de trabajo. Esta vista evita interpretar Silver como un único dataset cuando en realidad es una suma de entidades con grano distinto.

In [3]:
def infer_silver_source(path: Path) -> str:
    name = path.name.lower()
    if 'tweets' in name:
        return 'tweets'
    if 'noticias' in name or 'news' in name:
        return 'news'
    return 'unknown'

frames = []
for file_path in silver_files:
    df = pd.read_parquet(file_path)
    df['source_file'] = file_path.name
    df['source_group'] = infer_silver_source(file_path)
    frames.append(df)

if frames:
    silver_raw = pd.concat(frames, ignore_index=True)
else:
    silver_raw = pd.DataFrame()

print(f'Forma de silver_raw: {silver_raw.shape}')
silver_raw.head(5)

Forma de silver_raw: (408, 179)


,newsLink,comments,_id,snapshot_id,snapshot_date,source_file,source_group,type,id,url,...,quoted_tweet_quoted_tweet_retweeted_tweet,quoted_tweet_quoted_tweet_isLimitedReply,quoted_tweet_quoted_tweet_communityInfo,quoted_tweet_quoted_tweet_article,author_profile_bio_entities_description_symbols,entities_timestamps,quoted_tweet_author_profile_bio_entities_description_hashtags,quoted_tweet_author_profile_bio_entities_url_urls,text_cleaned,text_length
0,https://www.gsmarena.com/xiaomi_unveils_smart_...,"[""I like how at least some still make earbuds ...",6a19f676d5d72db4fa2cf569,6a19f676d5d72db4fa2cf568,1780086390412,noticias_processed_20260530_012014.parquet,news,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
1,https://www.gsmarena.com/samsung_is_now_shippi...,"[""Ai sux"", ""Ai will ruin this planet"", ""AI wil...",6a19f676d5d72db4fa2cf56a,6a19f676d5d72db4fa2cf568,1780086390412,noticias_processed_20260530_012014.parquet,news,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
2,https://www.gsmarena.com/samsung_galaxy_z_fold...,"[""Waiting for the day Samsung increases batter...",6a19f676d5d72db4fa2cf56b,6a19f676d5d72db4fa2cf568,1780086390412,noticias_processed_20260530_012014.parquet,news,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
3,https://www.gsmarena.com/vertu_unveils_alphafo...,"[""Rebranded zte with leather slapped onto it. ...",6a19f676d5d72db4fa2cf56c,6a19f676d5d72db4fa2cf568,1780086390412,noticias_processed_20260530_012014.parquet,news,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
4,https://www.gsmarena.com/hmd_grand_renders_and...,"[""-, 28 May 20261. How did you get banned in u...",6a19f676d5d72db4fa2cf56d,6a19f676d5d72db4fa2cf568,1780086390412,noticias_processed_20260530_012014.parquet,news,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN


## 3. Expansión del grano y consolidación de entidades

Silver no se debe leer como una sola tabla plana. Aquí se expanden tweets, noticias y comentarios para ver qué campos sobreviven en cada familia y qué información conviene conservar como canónica para Gold.

## 2. Auditoría de columnas y familias de datos

Aquí se mide cobertura, tipo y cardinalidad para separar columnas de negocio de columnas de soporte. La meta no es conservar todo: es identificar qué campos pueden sostener Gold sin inflar el modelo.

In [4]:
def is_missing_value(value) -> bool:
    try:
        return bool(pd.isna(value))
    except Exception:
        return False


def safe_nunique(series: pd.Series) -> int:
    normalized = series.map(lambda value: None if is_missing_value(value) else repr(value))
    return int(normalized.dropna().nunique(dropna=True))


if silver_raw.empty:
    audit = pd.DataFrame(columns=['column', 'dtype', 'missing_ratio', 'non_null_count', 'nunique'])
else:
    audit = pd.DataFrame({
        'column': silver_raw.columns,
        'dtype': silver_raw.dtypes.astype(str).values,
        'missing_ratio': silver_raw.isna().mean().values,
        'non_null_count': silver_raw.notna().sum().values,
        'nunique': [safe_nunique(silver_raw[c]) for c in silver_raw.columns],
    }).sort_values(['missing_ratio', 'column'], ascending=[False, True])

display(audit.head(80))

,column,dtype,missing_ratio,non_null_count,nunique
65,article,object,1.000000,0,0
58,author_automatedBy,object,1.000000,0,0
49,author_hasCustomTimelines,"datetime64[us, UTC]",1.000000,0,0
35,author_verifiedType,object,1.000000,0,0
59,card,float64,1.000000,0,0
...,...,...,...,...,...
116,quoted_tweet_author_canDm,object,0.845588,63,2
117,quoted_tweet_author_canMediaTag,object,0.845588,63,2
110,quoted_tweet_author_coverPicture,str,0.845588,63,18
118,quoted_tweet_author_createdAt,str,0.845588,63,19


### Lectura ejecutiva de la auditoría

- **Hallazgo principal:** la cobertura no es homogénea porque Silver mezcla familias de datos con propósitos distintos.
- **Dato relevante:** las columnas con alta cardinalidad o bajo completado suelen aportar trazabilidad, pero no siempre deben avanzar a Gold.
- **Decisión:** la primera selección debe priorizar columnas estables, repetibles y alineadas con una sola entidad de negocio.

In [5]:
try:
    from langdetect import DetectorFactory, LangDetectException, detect
    DetectorFactory.seed = 42
    LANGDETECT_AVAILABLE = True
except Exception:
    LANGDETECT_AVAILABLE = False


def first_existing(columns: list[str], candidates: list[str]) -> list[str]:
    available = set(columns)
    return [c for c in candidates if c in available]


def clean_text(text: str | None) -> str:
    if text is None:
        return ''
    cleaned = str(text)
    cleaned = re.sub(r'https?://\S+|www\.\S+', ' ', cleaned)
    cleaned = re.sub(r'@\w+', ' ', cleaned)
    cleaned = re.sub(r'#', ' ', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned)
    return cleaned.strip()


def detect_language_safe(text: str) -> str:
    if not text:
        return 'unknown'
    if not LANGDETECT_AVAILABLE:
        return 'unknown'
    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'


def to_text(value) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    if isinstance(value, str):
        return value
    if isinstance(value, (list, tuple, set)):
        return ' '.join(str(v) for v in value if v is not None)
    if hasattr(value, 'tolist') and not isinstance(value, (str, bytes)):
        as_list = value.tolist()
        if isinstance(as_list, list):
            return ' '.join(str(v) for v in as_list if v is not None)
        return str(as_list)
    return str(value)


def parse_mixed_datetime(series: pd.Series) -> pd.Series:
    as_text = series.map(to_text)
    numeric = pd.to_numeric(as_text, errors='coerce')
    numeric_ratio = float(numeric.notna().mean()) if len(numeric) else 0.0
    if numeric_ratio > 0.8:
        return pd.to_datetime(numeric, errors='coerce', unit='ms')
    return pd.to_datetime(as_text, errors='coerce', format='mixed')


def flatten_silver(df: pd.DataFrame) -> pd.DataFrame:
    parts = []

    if 'tweets' in df.columns:
        tweet_base = df[df['tweets'].notna()].copy()
        if not tweet_base.empty:
            tweet_base = tweet_base.explode('tweets', ignore_index=True)
            tweet_info = pd.json_normalize(tweet_base['tweets'], sep='_').add_prefix('tweet_')
            tweet_flat = pd.concat([tweet_base.drop(columns=['tweets']), tweet_info], axis=1)
            tweet_flat['record_type'] = 'tweet'
            parts.append(tweet_flat)

    if 'news' in df.columns:
        news_base = df[df['news'].notna()].copy()
        if not news_base.empty:
            news_base = news_base.explode('news', ignore_index=True)
            news_info = pd.json_normalize(news_base['news'], sep='_').add_prefix('news_')
            news_flat = pd.concat([news_base.drop(columns=['news']), news_info], axis=1)
            news_flat['record_type'] = 'news'
            parts.append(news_flat)

            if 'news_comments' in news_flat.columns:
                comments_flat = news_flat.explode('news_comments', ignore_index=True).copy()
                comments_flat['comment_text'] = comments_flat['news_comments'].map(to_text)
                comments_flat['record_type'] = 'news_comment'
                parts.append(comments_flat)

    if not parts:
        fallback = df.copy()
        fallback['record_type'] = 'snapshot'
        return fallback

    return pd.concat(parts, ignore_index=True)


silver_expanded = flatten_silver(silver_raw)
print(f'Forma de silver_expanded: {silver_expanded.shape}')

text_candidates = [
    'tweet_text', 'comment_text', 'news_comments', 'news_newsLink', 'text',
    'clean_text', 'comments', 'title', 'headline', 'content', 'news_title', 'news_text'
]
date_candidates = [
    'createdAt', 'updatedAt', 'date', 'date_parsed', 'tweet_createdAt',
    'tweet_created_at', 'published_at', 'timestamp'
]
id_candidates = [
    'id', '_id', 'tweet_id', 'news_id', 'news__id', 'source_file', 'record_type'
]

base_cols = first_existing(list(silver_expanded.columns), id_candidates)
base_cols += first_existing(list(silver_expanded.columns), date_candidates)
base_cols += first_existing(list(silver_expanded.columns), text_candidates)

metric_cols = [
    c for c in silver_expanded.columns
    if any(k in c.lower() for k in ['like', 'reply', 'retweet', 'quote', 'view', 'score', 'sentiment'])
]
meta_cols = [
    c for c in silver_expanded.columns
    if any(k in c.lower() for k in ['author', 'user', 'brand', 'source', 'lang'])
]

keep_cols = list(dict.fromkeys(base_cols + metric_cols + meta_cols))
silver_eda = silver_expanded[keep_cols].copy() if keep_cols else silver_expanded.copy()

text_cols_existing = first_existing(list(silver_eda.columns), text_candidates)
if text_cols_existing:
    text_frame = silver_eda[text_cols_existing].copy().map(to_text)
    silver_eda['raw_text'] = text_frame.replace('', pd.NA).bfill(axis=1).iloc[:, 0].fillna('').astype('string')
else:
    object_like_cols = [c for c in silver_eda.columns if silver_eda[c].dtype == 'object']
    if object_like_cols:
        fallback = silver_eda[object_like_cols[0]].map(to_text)
        silver_eda['raw_text'] = fallback.astype('string')
    else:
        silver_eda['raw_text'] = ''

silver_eda['clean_text'] = silver_eda['raw_text'].map(clean_text)
silver_eda['text_length'] = silver_eda['clean_text'].str.len()
silver_eda['detected_lang'] = silver_eda['clean_text'].fillna('').map(detect_language_safe)
silver_eda['has_text'] = silver_eda['clean_text'].str.len().fillna(0).gt(0)

date_cols_existing = first_existing(list(silver_eda.columns), date_candidates)
if date_cols_existing:
    parsed_series = None
    for col in date_cols_existing:
        candidate = parse_mixed_datetime(silver_eda[col])
        parsed_series = candidate if parsed_series is None else parsed_series.fillna(candidate)
    silver_eda['parsed_datetime'] = parsed_series
    silver_eda['event_date'] = silver_eda['parsed_datetime'].dt.date.astype('string')
else:
    silver_eda['parsed_datetime'] = pd.NaT
    silver_eda['event_date'] = pd.Series([''] * len(silver_eda), dtype='string')

dedupe_keys = [c for c in ['source_file', 'record_type', 'tweet_id', 'news_id', 'news__id', '_id', 'id', 'clean_text', 'event_date'] if c in silver_eda.columns]
before_dedup = len(silver_eda)
if dedupe_keys:
    silver_eda = silver_eda.drop_duplicates(subset=dedupe_keys).reset_index(drop=True)
after_dedup = len(silver_eda)

print(f'Registros antes de deduplicar: {before_dedup}')
print(f'Registros después de deduplicar: {after_dedup}')
print(f'Columnas en silver_eda: {len(silver_eda.columns)}')
print(f'Registros con texto: {int(silver_eda["has_text"].sum())}')
silver_eda.head(5)

Forma de silver_expanded: (408, 180)
Registros antes de deduplicar: 408
Registros después de deduplicar: 408
Columnas en silver_eda: 162
Registros con texto: 408


,id,_id,source_file,record_type,createdAt,text,comments,retweetCount,replyCount,likeCount,...,author_profile_bio_entities_description_user_mentions,author_profile_bio_entities_description_hashtags,author_profile_bio_entities_description_symbols,raw_text,clean_text,text_length,detected_lang,has_text,parsed_datetime,event_date
0,NaN,6a19f676d5d72db4fa2cf569,noticias_processed_20260530_012014.parquet,snapshot,NaN,NaN,"[""I like how at least some still make earbuds ...",NaN,NaN,NaN,...,NaN,NaN,NaN,"[""I like how at least some still make earbuds ...","[""I like how at least some still make earbuds ...",1735,unknown,True,NaT,<NA>
1,NaN,6a19f676d5d72db4fa2cf56a,noticias_processed_20260530_012014.parquet,snapshot,NaN,NaN,"[""Ai sux"", ""Ai will ruin this planet"", ""AI wil...",NaN,NaN,NaN,...,NaN,NaN,NaN,"[""Ai sux"", ""Ai will ruin this planet"", ""AI wil...","[""Ai sux"", ""Ai will ruin this planet"", ""AI wil...",280,unknown,True,NaT,<NA>
2,NaN,6a19f676d5d72db4fa2cf56b,noticias_processed_20260530_012014.parquet,snapshot,NaN,NaN,"[""Waiting for the day Samsung increases batter...",NaN,NaN,NaN,...,NaN,NaN,NaN,"[""Waiting for the day Samsung increases batter...","[""Waiting for the day Samsung increases batter...",72,unknown,True,NaT,<NA>
3,NaN,6a19f676d5d72db4fa2cf56c,noticias_processed_20260530_012014.parquet,snapshot,NaN,NaN,"[""Rebranded zte with leather slapped onto it. ...",NaN,NaN,NaN,...,NaN,NaN,NaN,"[""Rebranded zte with leather slapped onto it. ...","[""Rebranded zte with leather slapped onto it. ...",2402,unknown,True,NaT,<NA>
4,NaN,6a19f676d5d72db4fa2cf56d,noticias_processed_20260530_012014.parquet,snapshot,NaN,NaN,"[""-, 28 May 20261. How did you get banned in u...",NaN,NaN,NaN,...,NaN,NaN,NaN,"[""-, 28 May 20261. How did you get banned in u...","[""-, 28 May 20261. How did you get banned in u...",4303,unknown,True,NaT,<NA>


### Lectura ejecutiva de la normalización

- **Hallazgo principal:** Silver ya quedó expandido por entidad y no como una tabla monolítica.
- **Dato relevante:** el texto canónico, la fecha y el tipo de registro son los ejes más útiles para EDA y para definir Gold.
- **Decisión:** la siguiente capa debe seleccionar columnas con cobertura estable y evitar joins que mezclen grano sin necesidad.

In [6]:
silver_eda.to_csv(SILVER_EDA_CSV, index=False, encoding='utf-8')
print(f'CSV guardado en: {SILVER_EDA_CSV}')
print(f'Registros exportados: {len(silver_eda)}')
print(f'Columnas exportadas: {len(silver_eda.columns)}')

CSV guardado en: /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming/notebooks/output/silver_eda.csv
Registros exportados: 408
Columnas exportadas: 162


## 4. Resumen ejecutivo para EDA y próximo paso Gold

El CSV exportado ya resume las variables de trabajo de Silver. Desde aquí el EDA puede revisar nulos, distribuciones, outliers, cobertura por tipo de registro y estabilidad temporal.

La decisión pendiente es separar Gold por entidad: tweets como base, y noticias o comentarios como capas específicas solo si el caso de uso lo justifica.

In [7]:
summary = {
    'rows': len(silver_eda),
    'columns': len(silver_eda.columns),
    'non_empty_text_ratio': float(silver_eda['has_text'].mean()) if 'has_text' in silver_eda.columns and len(silver_eda) else 0.0,
}
print(summary)

if 'detected_lang' in silver_eda.columns:
    display(silver_eda['detected_lang'].value_counts().head(10).to_frame('rows'))

if 'source_file' in silver_eda.columns:
    display(silver_eda['source_file'].value_counts().head(10).to_frame('rows'))

missing_top = silver_eda.isna().mean().sort_values(ascending=False).head(20).to_frame('missing_ratio')
display(missing_top)

{'rows': 408, 'columns': 162, 'non_empty_text_ratio': 1.0}


,rows
detected_lang,
unknown,408


,rows
source_file,
tweets_processed_20260530_012013.parquet,60
tweets_processed_20260601_204305.parquet,60
tweets_processed_20260601_204414.parquet,60
tweets_processed_20260601_204416.parquet,60
tweets_processed_20260602_223240.parquet,60
tweets_processed_20260602_223252.parquet,60
noticias_processed_20260530_012014.parquet,8
noticias_processed_20260601_204305.parquet,7
noticias_processed_20260601_204418.parquet,7


,missing_ratio
quoted_tweet,1.0
retweeted_tweet,1.0
inReplyToId,1.0
quoted_tweet_quoted_tweet_card,1.0
quoted_tweet_quoted_tweet_inReplyToUsername,1.0
quoted_tweet_quoted_tweet_inReplyToUserId,1.0
quoted_tweet_entities_timestamps,1.0
quoted_tweet_article,1.0
quoted_tweet_communityInfo,1.0
quoted_tweet_retweeted_tweet,1.0


## 5. Validación de nulos: estructurales vs reales

Estas celdas verifican si los nulos vienen por columnas no aplicables según el tipo de registro (`tweet`, `news`, `news_comment`) o si son faltantes reales en campos que deberían existir.

In [8]:
if not silver_raw.empty:
    groups = silver_raw.groupby('source_file', dropna=False)
    rows = groups.size().rename('rows')
    if 'tweets' in silver_raw.columns:
        non_null_tweets = groups['tweets'].apply(lambda s: int(s.notna().sum())).rename('non_null_tweets')
    else:
        non_null_tweets = pd.Series(0, index=rows.index, name='non_null_tweets')
    if 'news' in silver_raw.columns:
        non_null_news = groups['news'].apply(lambda s: int(s.notna().sum())).rename('non_null_news')
    else:
        non_null_news = pd.Series(0, index=rows.index, name='non_null_news')

    source_level_quality = pd.concat([rows, non_null_tweets, non_null_news], axis=1).reset_index()
else:
    source_level_quality = pd.DataFrame(columns=['source_file', 'rows', 'non_null_tweets', 'non_null_news'])

display(source_level_quality)

record_type_quality = (
    silver_eda.groupby('record_type')
    .agg(
        rows=('record_type', 'size'),
        has_text_ratio=('has_text', 'mean'),
        null_text_ratio=('raw_text', lambda s: float((s.fillna('') == '').mean())),
        null_datetime_ratio=('parsed_datetime', lambda s: float(s.isna().mean())),
    )
    .reset_index()
)

display(record_type_quality)

core_fields = [c for c in ['tweet_id', 'news__id', 'comment_text', 'tweet_text', 'news_newsLink'] if c in silver_eda.columns]
if core_fields:
    core_nulls = (
        silver_eda.groupby('record_type')[core_fields]
        .apply(lambda df: df.isna().mean())
        .reset_index()
    )
    display(core_nulls)

,source_file,rows,non_null_tweets,non_null_news
0,noticias_processed_20260530_012014.parquet,8,0,0
1,noticias_processed_20260601_204305.parquet,7,0,0
2,noticias_processed_20260601_204418.parquet,7,0,0
3,noticias_processed_20260601_204419.parquet,7,0,0
4,noticias_processed_20260602_014739.parquet,7,0,0
5,noticias_processed_20260602_223238.parquet,7,0,0
6,noticias_processed_20260602_223251.parquet,5,0,0
7,tweets_processed_20260530_012013.parquet,60,0,0
8,tweets_processed_20260601_204305.parquet,60,0,0
9,tweets_processed_20260601_204414.parquet,60,0,0


,record_type,rows,has_text_ratio,null_text_ratio,null_datetime_ratio
0,snapshot,408,1.0,0.0,0.117647


### Lectura ejecutiva de la validación

- **Hallazgo principal:** la mayor parte de los nulos responde al grano de cada entidad, no a un error de lectura.
- **Dato relevante:** cuando una columna solo aplica a tweets o solo a noticias, el nulo en la otra familia es esperado.
- **Decisión:** antes de modelar Gold hay que separar nulos estructurales de faltantes reales para no castigar campos que sí son válidos.

In [9]:
files_reloaded = []
for file_path in silver_files:
    test_df = pd.read_parquet(file_path)
    files_reloaded.append({
        'source_file': file_path.name,
        'rows_read': len(test_df),
        'columns_read': len(test_df.columns),
        'read_ok': True,
    })

read_validation = pd.DataFrame(files_reloaded)

total_rows_raw = int(silver_raw.groupby('source_file').size().sum())
total_rows_reloaded = int(read_validation['rows_read'].sum())

print(f'Total de filas desde el consolidado bruto: {total_rows_raw}')
print(f'Total de filas desde la recarga directa: {total_rows_reloaded}')
print(f'Coincidencia de filas: {total_rows_raw == total_rows_reloaded}')

display(read_validation.sort_values('source_file'))

Total de filas desde el consolidado bruto: 408
Total de filas desde la recarga directa: 408
Coincidencia de filas: True


,source_file,rows_read,columns_read,read_ok
0,noticias_processed_20260530_012014.parquet,8,5,True
1,noticias_processed_20260601_204305.parquet,7,5,True
2,noticias_processed_20260601_204418.parquet,7,5,True
3,noticias_processed_20260601_204419.parquet,7,5,True
4,noticias_processed_20260602_014739.parquet,7,5,True
5,noticias_processed_20260602_223238.parquet,7,5,True
6,noticias_processed_20260602_223251.parquet,5,5,True
7,tweets_processed_20260530_012013.parquet,60,173,True
8,tweets_processed_20260601_204305.parquet,60,173,True
9,tweets_processed_20260601_204414.parquet,60,173,True
